In [4]:
import pyarrow as pa, pyarrow.parquet as pq
import pandas as pd
from pathlib import Path
PQ = Path("db/parquet"); PQ.mkdir(parents=True, exist_ok=True)

In [12]:
orders = pd.read_csv("data/fact_orders.csv")
items = pd.read_csv("data/fact_order_items.csv")
orders.dtypes

order_id        int64
customer_id     int64
order_ts       object
status         object
dtype: object

In [10]:
orders["status"] = orders.status.astype("category")
orders.to_parquet(PQ / "fact_orders.parquet")
items.to_parquet(PQ / "fact_order_items.parquet")          # items read with dtype={"discount_pct": "Int64"}
pd.read_parquet(PQ / "fact_orders.parquet").head()


,order_id,customer_id,order_ts,status
0,1,40,2025-05-28 14:51:36,shipped
1,2,161,2025-03-03 09:18:50,completed
2,3,116,2025-01-08 07:15:44,completed
3,4,27,2025-06-07 11:22:45,cancelled
4,5,157,2025-01-31 19:48:39,completed


In [11]:
pd.read_parquet(PQ / "fact_orders.parquet").dtypes

order_id          int64
customer_id       int64
order_ts         object
status         category
dtype: object

In [14]:
items.to_parquet(PQ / "x.parquet", compression="zstd")

In [20]:
pd.read_parquet(PQ / "fact_orders.parquet",
                columns= ["order_id", "customer_id"],
                 filters=[("status", "==", "refunded")])


,order_id,customer_id
0,15,93
1,20,71
2,37,168
3,62,86
4,71,81
...,...,...
64,1395,114
65,1437,78
66,1464,19
67,1471,71


In [21]:
pq.read_schema(PQ / "fact_orders.parquet")

order_id: int64
customer_id: int64
order_ts: string
status: dictionary<values=string, indices=int8, ordered=0>
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 777

In [23]:
md = pq.read_metadata(PQ / "fact_orders.parquet")
md

  created_by: parquet-cpp-arrow version 23.0.1
  num_columns: 4
  num_rows: 1500
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 3119

In [39]:
t = pa.Table.from_pandas(items)
t.schema

order_item_id: int64
order_id: int64
product_id: int64
quantity: int64
unit_price: double
discount_pct: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 992

In [38]:
t.column("discount_pct")

[
  [
    10,
    0,
    5,
    0,
    0,
    ...
    0,
    null,
    5,
    null,
    0
  ]
]

In [34]:
t.to_pandas().head()

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct
0,1,1,99,1,473.17,10.0
1,2,2,57,1,291.26,0.0
2,3,3,10,3,596.95,5.0
3,4,3,83,1,560.17,0.0
4,5,3,111,1,442.62,0.0


In [40]:
import pyarrow.compute as pc
pc.sum(t["quantity"]).as_py()

3468

In [41]:
import duckdb

In [42]:
duckdb.sql(f"select count(*), sum(quantity) from '{PQ}/fact_order_items.parquet'")

┌──────────────┬───────────────┐
│ count_star() │ sum(quantity) │
│    int64     │    int128     │
├──────────────┼───────────────┤
│         2643 │          3468 │
└──────────────┴───────────────┘

In [43]:
duckdb.sql(f"""COPY (select status, count(*) as n from '{PQ}/fact_orders.parquet' group by 1)
               TO '{PQ}/status.parquet' (FORMAT PARQUET)""")

In [44]:
duckdb.sql(f"COPY (select * from 'data/fact_orders.csv') TO '{PQ}/o.parquet' (FORMAT PARQUET, COMPRESSION zstd)")   # CSV -> Parquet, 14.1 KB

In [45]:
duckdb.sql(f"select * from parquet_metadata('{PQ}/fact_orders.parquet')")  

┌────────────────────────────────┬──────────────┬────────────────────┬───────────────────────┬─────────────────┬───────────┬─────────────┬────────────┬────────────────┬────────────┬───────────┬───────────┬──────────────────┬──────────────────────┬─────────────────────┬─────────────────────┬─────────────┬────────────────────────────┬───────────────────┬────────────────────────┬──────────────────┬───────────────────────┬─────────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬──────────────┬──────────────┬────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│           file_name            │ row_group_id │ row_group_num_rows │ row_group_num_columns │ row_group_bytes │ column_id │ file_offset │ num_values │ path_in_schema │    type    │ stats_min │ stats_max │ stats_null_count │ stats_distinct_count │   stats_min_value   │   stats_max_value   │ compressio

In [61]:
DATA = Path("data")
orders    = pd.read_csv(DATA / "fact_orders.csv", parse_dates=["order_ts"])

In [62]:
PQ = Path("db/parquet")
orders["month"] = orders.order_ts.dt.strftime("%Y-%m")
orders.to_parquet(PQ / "orders_by_month", partition_cols=["month"])
orders.groupby("month").count()

,order_id,customer_id,order_ts,status
month,,,,
2025-01,249,249,249,249
2025-02,236,236,236,236
2025-03,264,264,264,264
2025-04,242,242,242,242
2025-05,270,270,270,270
2025-06,239,239,239,239


In [63]:
pd.read_parquet(PQ / "orders_by_month")

,order_id,customer_id,order_ts,status,month
0,3,116,2025-01-08 07:15:44,completed,2025-01
1,5,157,2025-01-31 19:48:39,completed,2025-01
2,8,36,2025-01-31 02:05:48,completed,2025-01
3,10,65,2025-01-11 03:10:31,completed,2025-01
4,15,93,2025-01-14 03:24:30,refunded,2025-01
...,...,...,...,...,...
1495,1477,117,2025-06-05 02:18:24,completed,2025-06
1496,1484,66,2025-06-07 13:01:35,completed,2025-06
1497,1489,128,2025-06-07 02:02:32,completed,2025-06
1498,1494,49,2025-06-15 02:43:35,shipped,2025-06


In [64]:
pd.read_parquet(PQ / "orders_by_month", filters=[("month", "==", "2025-03")])

,order_id,customer_id,order_ts,status,month
0,2,161,2025-03-03 09:18:50,completed,2025-03
1,19,21,2025-03-05 16:13:03,completed,2025-03
2,26,197,2025-03-21 12:05:54,shipped,2025-03
3,31,28,2025-03-03 23:11:16,completed,2025-03
4,35,131,2025-03-19 22:35:57,completed,2025-03
...,...,...,...,...,...
259,1461,159,2025-03-19 19:17:54,completed,2025-03
260,1472,98,2025-03-18 10:34:15,shipped,2025-03
261,1481,163,2025-03-25 06:58:51,completed,2025-03
262,1488,17,2025-03-13 01:33:19,completed,2025-03


In [ ]:
con = duckdb.connect()
con.sql("INSTALL sqlite; LOAD sqlite; ATTACH 'db/marketplace.db' AS m (TYPE sqlite, READ_ONLY)")
lake = PQ / "lake"; lake.mkdir(exist_ok=True)
tables = [r[0] for r in con.sql("select table_name from information_schema.tables where table_catalog = 'm'").fetchall()]
for t in tables:
    con.sql(f"COPY (select * from m.{t}) TO '{lake}/{t}.parquet' (FORMAT PARQUET, COMPRESSION zstd)")